In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [2]:
X_train = pd.read_csv('../data/X_train.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
X_test = pd.read_csv('../data/X_test.csv')
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [3]:
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Attack ratio in train: {y_train.mean():.2%}")

Train: (553124, 78), Test: (138282, 78)
Attack ratio in train: 36.41%


In [4]:
from sklearn.utils.class_weight import compute_class_weight


In [5]:
class_weights = compute_class_weight('balanced',
                                      classes=np.unique(y_train),
                                      y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights: {class_weight_dict}")

Class weights: {0: np.float64(0.7862548543551312), 1: np.float64(1.3733476347962539)}


In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight=class_weight_dict,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)
pickle.dump(rf_model, open('../results/random_forest_model.pkl', 'wb'))
print("Random Forest trained and saved.")

Random Forest trained and saved.


In [10]:
scale_pos = class_weight_dict[1] / class_weight_dict[0]

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    eval_metric='logloss', device = 'cuda'
)

xgb_model.fit(X_train, y_train)
pickle.dump(xgb_model, open('../results/xgboost_model.pkl', 'wb'))
print("XGBoost trained and saved.")

XGBoost trained and saved.


In [9]:
def evaluate(model_name, y_true, y_pred):
    print(f"\n{'='*40}")
    print(f"Model: {model_name}")
    print(f"{'='*40}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
    print(f"\n{classification_report(y_true, y_pred)}")

y_pred_rf  = rf_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)

evaluate("Random Forest", y_test, y_pred_rf)
evaluate("XGBoost", y_test, y_pred_xgb)

c:\Users\Maneshna\miniconda3\envs\torch_env\lib\site-packages\xgboost\core.py:751: UserWarning: [23:43:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



Model: Random Forest
Accuracy:  0.9995
Precision: 0.9988
Recall:    0.9998
F1 Score:  0.9993

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     87937
           1       1.00      1.00      1.00     50345

    accuracy                           1.00    138282
   macro avg       1.00      1.00      1.00    138282
weighted avg       1.00      1.00      1.00    138282


Model: XGBoost
Accuracy:  0.9998
Precision: 0.9995
Recall:    1.0000
F1 Score:  0.9997

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     87937
           1       1.00      1.00      1.00     50345

    accuracy                           1.00    138282
   macro avg       1.00      1.00      1.00    138282
weighted avg       1.00      1.00      1.00    138282

